# Subagents — Claude Managed Agents (CLI)

The same flow as [`06-subagents/`](../06-subagents/), with the **`ant`** CLI: a **lead** agent splits a contract review between two **specialist** subagents, each working in its own **session thread**.

The CLI adds one nice twist: **resources refer to each other by path**. The lead's `agent.md` lists its subagents as `./liability-reviewer.md` and `./renewal-reviewer.md`, and `ant apply` creates them in dependency order, fills in their real IDs, and pins their versions.

> Prerequisite: the `ant` CLI (>= 1.40) installed and authenticated — see [`01-basics-cli/`](../01-basics-cli/).

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so a bit of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON output in
#    ANSI color codes and break parsing — clear it so the output stays plain;
#  - `!` runs commands in a pseudo-terminal, so ant probes it with escape
#    sequences that leak into the cell output — TERM=dumb turns that off.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

In [ ]:
import json
import subprocess
import time


def ant(*args: str, input: str | None = None) -> str:
    """Run an ant command and return its stdout; show stderr if it fails."""
    result = subprocess.run(["ant", *args], input=input, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result.stdout


def ant_json(*args: str):
    return json.loads(ant(*args, "--format", "json"))


def ant_jsonl(*args: str) -> list:
    return [json.loads(line) for line in ant(*args, "--format", "jsonl").splitlines() if line.strip()]


def send(session_id: str, *events: dict) -> list:
    """Send events; returns the queued events (with their ids)."""
    return json.loads(ant("beta:sessions:events", "send", "--session-id", session_id, "--format", "json",
                          input=json.dumps({"events": list(events)})))["data"]


def open_stream(session_id: str):
    return subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "json"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )


def events(stream):
    """Yield each event from `ant ... stream --format json` as a dict."""
    buf = []
    for line in stream.stdout:
        buf.append(line)
        if line.rstrip("\n") == "}":
            yield json.loads("".join(buf))
            buf = []


def message(text: str) -> dict:
    return {"type": "user.message", "content": [{"type": "text", "text": text}]}


def text_of(event: dict) -> str:
    return "".join(b["text"] for b in event.get("content", []) if b.get("type") == "text")


def cents(amount: dict) -> str:
    return f"${int(amount['amount']) / 100:.2f}"

## 2. Declare the Specialists and the Lead

The specialists are ordinary agents on `claude-haiku-5-5`, with no `multiagent` block: delegation is one level deep, so an agent that has one can't be a subagent.

The lead's frontmatter holds the `multiagent` block:
- `subagents.predefined_agents` lists the specialists **by path**;
- `subagents.inline_agents: disabled` means the lead can't define its own subagents on the fly, only use ours;
- `workflows: disabled` turns off dynamic workflows, which are on by default (module 07).

> **Names matter.** The lead delegates to a subagent by its **`name`**. Use exactly the names from the system prompt. If they don't match, the lead can't find its subagents and quietly does the work itself.

In [ ]:
%%writefile liability-reviewer.md
---
name: Liability Reviewer
description: Reviews liability caps, indemnities and exclusions in supplier contracts.
model: claude-haiku-5-5
tools:
  - type: agent_toolset_20260401
    configs:
      - name: web_search
        enabled: false
      - name: web_fetch
        enabled: false
---

You review supplier contracts for Brightloom Inc., focusing only on liability:
liability caps, carve-outs from the cap, indemnities, and exclusions of damages.
For each contract, report the cap, what is excluded from it, and one risk for Brightloom.
Quote clause numbers. Be concise.

In [ ]:
%%writefile renewal-reviewer.md
---
name: Renewal Reviewer
description: Reviews term, renewal, notice and termination clauses in supplier contracts.
model: claude-haiku-5-5
tools:
  - type: agent_toolset_20260401
    configs:
      - name: web_search
        enabled: false
      - name: web_fetch
        enabled: false
---

You review supplier contracts for Brightloom Inc., focusing only on term and exit:
initial term, automatic renewal, non-renewal notice periods, and termination rights.
For each contract, report the next date by which Brightloom must act to avoid an unwanted
renewal, if any (use the sandbox clock for today's date). Quote clause numbers. Be concise.

In [ ]:
%%writefile agent.md
---
name: Contract Review Lead (CLI)
model: claude-opus-5-5
tools:
  - type: agent_toolset_20260401
multiagent:
  type: multiagent_20261001
  subagents:
    type: enabled
    inline_agents:
      type: disabled
    predefined_agents:
      - ./liability-reviewer.md
      - ./renewal-reviewer.md
  workflows:
    type: disabled
---

You lead contract reviews for Brightloom Inc. Contracts are mounted read-only under
/mnt/session/uploads/contracts/. Delegate the liability review to the Liability Reviewer and
the renewal review to the Renewal Reviewer, in parallel. Each specialist can read the files
itself; tell it where they are. Then merge their findings into one memo at
/mnt/session/outputs/memo.md with a short ranked list of actions. Keep chat replies short.

In [ ]:
%%writefile environment.yaml
# yaml-language-server: $schema=https://platform.claude.com/schemas/ant/beta/environment.json
name: subagents-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

Apply **only the lead and the environment**. `ant apply` follows the lead's references, creates both specialists first, and then creates the lead with their real IDs. The lead stays pinned to the specialists' current versions. If you edit a specialist and apply again, `ant apply` updates the lead's list in the same run.

In [ ]:
!ant apply agent.md environment.yaml --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))["resources"]
env_id = lock["./environment.yaml"]["id"]
lead_id, lead_version = lock["./agent.md"]["id"], int(lock["./agent.md"]["version"])
for path in ("./liability-reviewer.md", "./renewal-reviewer.md", "./agent.md"):
    print(f"{path:<24} {lock[path]['id']}  v{lock[path]['version']}")

print(json.dumps(ant_json("beta:agents", "retrieve", "--agent-id", lead_id, "--transform", "multiagent"), indent=2))

## 3. Mount the Contracts and Start a Budgeted Session

Same as module 04. All threads share one sandbox, and one budget covers every thread.

In [ ]:
from pathlib import Path

uploaded = {}
for path in sorted(Path("../sample-data/contracts").glob("*.md")):
    uploaded[path.name] = ant("files", "upload", "--file", str(path), "--transform", "id", "-r").strip()

agent_ref = json.dumps({"type": "agent", "id": lead_id, "version": lead_version})
budget = json.dumps({"type": "limit", "max_list_cost": {"amount": "300", "currency": "USD"}})
resource_flags = []
for name, file_id in uploaded.items():
    resource_flags += ["--resource", json.dumps({"type": "file", "file_id": file_id, "mount_path": f"/contracts/{name}"})]

session_id = ant("beta:sessions", "create", "--agent", agent_ref, "--environment-id", env_id,
                 "--title", "Contract review with subagents CLI", "--budget", budget,
                 *resource_flags, "--transform", "id", "-r").strip()
print(f"Session: {session_id}  ({len(uploaded)} contracts mounted)")

## 4. Follow the Primary Thread

The session stream is the **primary thread**. It shows the lead's own work, plus a condensed view of the subagents:

| Event | Meaning |
|---|---|
| `session.thread_created` | A subagent thread was created (`session_thread_id`, `agent_name`) |
| `agent.thread_message_sent` | The lead sent a subagent a task (`to_agent_name`) |
| `agent.thread_message_received` | A subagent reported back (`from_agent_name`) |
| `session.thread_status_idle` | **One thread** went idle |
| `session.status_idle` | The **whole session** went idle, meaning no thread is running. This is the only idle that ends the turn. |

A subagent's own tool calls stay on its thread. Calls that need your client are cross-posted here.

In [ ]:
def follow(stream) -> str:
    """Print the primary thread's events until the whole session ends its turn."""
    names = {}
    for event in events(stream):
        match event["type"]:
            case "session.thread_created":
                names[event["session_thread_id"]] = event["agent_name"]
                print(f"[thread created] {event['agent_name']}  ({event['session_thread_id']})")
            case "agent.thread_message_sent":
                print(f"[lead -> {event.get('to_agent_name')}] {text_of(event)[:160]}")
            case "agent.thread_message_received":
                print(f"[{event.get('from_agent_name')} -> lead] {text_of(event)[:300]}")
            case "session.thread_status_idle":
                print(f"[thread idle] {names.get(event.get('session_thread_id'), 'lead')}: {event['stop_reason']['type']}")
            case "agent.message":
                print(f"Lead: {text_of(event)}")
            case "agent.tool_use":
                i = event["input"]
                print(f"[lead {event['name']}] {str(i.get('command') or i.get('file_path') or '')[:80]}")
            case "session.status_idle":
                if event["stop_reason"]["type"] == "requires_action":
                    continue
                print(f"[session idle: {event['stop_reason']['type']}]")
                return event["stop_reason"]["type"]
            case "session.status_terminated":
                return "terminated"


stream = open_stream(session_id)
send(session_id, message("Review all the contracts and write the memo."))
follow(stream)
stream.terminate()

## 5. Inspect the Threads

`ant beta:sessions:threads list` returns every thread. The primary thread is the one with a null `parent_thread_id`. Each thread carries its own `usage`, priced at its own model.

`ant beta:sessions:threads:events list` drills into one thread's own history: the task it received, its tool calls, and its reply. (`... threads:events stream` follows it live.)

In [ ]:
threads = ant_jsonl("beta:sessions:threads", "list", "--session-id", session_id)
for t in threads:
    role = "primary" if t.get("parent_thread_id") is None else "child"
    cost = cents(t["usage"]["list_cost"]) if t.get("usage") and t["usage"].get("list_cost") else "-"
    print(f"{t['id']}  {role:<8} {t['agent'].get('name', t['agent']['type']):<28} {t['status']:<10} {cost}")

liability = next(t for t in threads if t["agent"].get("name", "").startswith("Liability Reviewer"))
print(f"\n--- inside {liability['agent']['name']} ---")
for event in ant_jsonl("beta:sessions:threads:events", "list", "--session-id", session_id, "--thread-id", liability["id"]):
    if event["type"] == "agent.thread_message_received":
        print(f"[task] {text_of(event)[:200]}")
    elif event["type"] == "agent.tool_use":
        print(f"[{event['name']}] {str(event['input'].get('command') or event['input'].get('file_path') or '')[:80]}")
    elif event["type"] == "agent.message":
        print(f"Reviewer: {text_of(event)[:300]}")

## 6. Follow Up with the Same Subagent

Subagent threads are **persistent**. When the lead sends a follow-up, it goes to the same thread, and the subagent answers with its earlier review still in context.

In [ ]:
stream = open_stream(session_id)
send(session_id, message(
    "Ask the Liability Reviewer, without re-reading the contracts: which single contract would it "
    "renegotiate first, and what exact clause change would it ask for? Relay its answer."
))
follow(stream)
stream.terminate()

## 7. Download the Memo, Archive the Threads

A session can have at most **25 child threads** at a time, and idle threads count until they're archived. `ant beta:sessions:threads archive` frees a thread's slot once its work is done.

In [ ]:
from IPython.display import Markdown, display

memo_id = None
for _ in range(20):   # outputs can lag the idle by a few seconds
    files = ant_jsonl("beta:files", "list", "--scope-id", session_id, "--beta", "managed-agents-2026-04-01")
    memo_id = next((f["id"] for f in files if f["downloadable"] and f["filename"] == "memo.md"), None)
    if memo_id:
        break
    time.sleep(3)

if memo_id:
    display(Markdown(ant("files", "download", "--file-id", memo_id, "--output", "-")))
else:
    print("memo.md not found in the session outputs")

In [ ]:
for t in ant_jsonl("beta:sessions:threads", "list", "--session-id", session_id):
    if t.get("parent_thread_id") and t["status"] == "idle":
        status = ant("beta:sessions:threads", "archive", "--session-id", session_id, "--thread-id", t["id"],
                     "--transform", "status", "-r").strip()
        print(f"Archived {t['agent'].get('name')}: {status}")

for file_id in uploaded.values():
    ant("files", "delete", "--file-id", file_id)

## Summary

```
liability-reviewer.md / renewal-reviewer.md   →  ordinary agents (Haiku), no multiagent block
agent.md  multiagent.subagents.predefined_agents: [./liability-reviewer.md, ...]   ← paths!
ant apply agent.md                            →  creates the specialists first, fills in IDs, pins versions
primary stream                                →  thread_created, thread_message_sent / _received, thread_status_*
ant beta:sessions:threads list / :events list / archive
```

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`06-subagents/`](../06-subagents/)) |
|---|---|---|
| Reference a subagent | a path (`./reviewer.md`) resolved by `ant apply` | `{"type": "agent", "id": ..., "version": ...}` |
| List threads | `ant beta:sessions:threads list` | `client.beta.sessions.threads.list()` |
| A thread's events | `ant beta:sessions:threads:events list` | `client.beta.sessions.threads.events.list()` |
| Archive a thread | `ant beta:sessions:threads archive` | `client.beta.sessions.threads.archive()` |

### Next steps
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel